# Kaggle GPU verification — IBM AML GNN

Targeted verification для курсовой: проверить ключевые standalone GNN на Kaggle GPU и пересобрать итоговые таблицы/графики. В текущем Kaggle PyTorch P100 может быть несовместима (sm_60), поэтому предпочтительно выбрать T4 x2 или L4.

Что запускается по умолчанию:
- `ibm_gine_fulldata`
- `ibm_multignn_fulldata`
- `ibm_pna_fulldata`
- `src.compare --ibm` для сборки Markdown/PNG

Опционально можно включить multi-seed и гибрид `GNN embedding -> XGBoost`. Для полного гибрида на 5.08M ребер нужно много времени и RAM; включайте его отдельным запуском.

## 0. Kaggle setup

Перед запуском:
1. Notebook settings -> Accelerator -> `GPU T4 x2` (рекомендуется) или `GPU L4`. P100 использовать только если установлен PyTorch с поддержкой sm_60.
2. Notebook settings -> Internet -> `On`, если репозиторий или PyG wheels не прикреплены как dataset.
3. Add Data -> IBM AML dataset с файлами `HI-Small_Trans.csv` и `HI-Small_Patterns.txt`.
4. Если репозиторий приватный и git clone не доступен, прикрепите zip/source как Kaggle Dataset и укажите `REPO_INPUT_DIR` ниже.

In [5]:
import os, sys, subprocess, shutil, json, glob, textwrap, pathlib
from pathlib import Path

import torch
assert torch.cuda.is_available(), "GPU не виден. Включи Accelerator -> GPU P100."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
name = torch.cuda.get_device_name(0)
if "P100" in name:
    print("WARNING: P100 (sm_60) может быть несовместима с текущим Kaggle PyTorch. Если видишь CUDA capability warning, переключись на T4 x2 или L4.")
if torch.cuda.device_count() > 1:
    print("GPUs visible:", torch.cuda.device_count(), "(код использует одну GPU; T4 x2 полезна главным образом совместимостью/доступностью, не удвоением памяти)")

GPU: Tesla T4
torch: 2.10.0+cu128 cuda: 12.8
GPUs visible: 2 (код использует одну GPU; T4 x2 полезна главным образом совместимостью/доступностью, не удвоением памяти)


## 1. Репозиторий

По умолчанию notebook клонирует GitHub repo. Если GitHub недоступен, загрузите исходники как Kaggle Dataset и задайте `REPO_INPUT_DIR`.

In [6]:
import os, shutil, zipfile, glob
from pathlib import Path

REPO = Path("/kaggle/working/gnn-aml-transaction-graphs")

def looks_like_repo(path: Path) -> bool:
    return (
        (path / "src" / "train_edge.py").exists()
        and (path / "src" / "compare.py").exists()
        and (path / "configs").exists()
    )

def copy_repo(src: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns(".git", "__pycache__", ".venv", "data", "wandb", ".pytest_cache")
    shutil.copytree(src, dst, ignore=ignore)

print("Kaggle inputs:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print(" -", p)

# 1) Найти уже распакованный repo среди attached Kaggle datasets.
candidates = []
for p in Path("/kaggle/input").glob("**"):
    if p.is_dir() and looks_like_repo(p):
        candidates.append(p)

if candidates:
    src = sorted(candidates, key=lambda x: len(str(x)))[0]
    copy_repo(src, REPO)
    print("copied repo from:", src)

else:
    # 2) Найти zip с repo среди attached Kaggle datasets.
    found = False
    for z in Path("/kaggle/input").glob("**/*.zip"):
        print("checking zip:", z)
        tmp = Path("/kaggle/working/_repo_zip_extract")
        if tmp.exists():
            shutil.rmtree(tmp)
        tmp.mkdir(parents=True, exist_ok=True)

        try:
            with zipfile.ZipFile(z) as zz:
                zz.extractall(tmp)
        except zipfile.BadZipFile:
            continue

        repo_dirs = [p for p in tmp.glob("**") if p.is_dir() and looks_like_repo(p)]
        if repo_dirs:
            src = sorted(repo_dirs, key=lambda x: len(str(x)))[0]
            copy_repo(src, REPO)
            print("extracted repo from zip:", z)
            found = True
            break

    if not found:
        print("\nНе найден код проекта в /kaggle/input.")
        print("Сейчас, вероятно, прикреплен только IBM AML dataset.")
        print("Нужно отдельно загрузить zip/folder проекта с src/, configs/, scripts/, results/.")
        print("\nПервые файлы в /kaggle/input для диагностики:")
        for f in list(Path("/kaggle/input").glob("**/*"))[:80]:
            print(" -", f)
        raise FileNotFoundError("Attach project source as Kaggle Dataset, not only AML data.")

os.chdir(REPO)
print("cwd:", Path.cwd())
print("repo ok:", looks_like_repo(REPO))
!ls -la

Kaggle inputs:
 - /kaggle/input/datasets
copied repo from: /kaggle/input/datasets/aleksandresenin/ibm-aml-data
cwd: /kaggle/working/gnn-aml-transaction-graphs
repo ok: True
total 84
drwxr-xr-x 11 root root  4096 Jun 23 08:15 .
drwxr-xr-x  4 root root  4096 Jun 23 11:35 ..
drwxr-xr-x  2 root root  4096 Jun 23 08:15 app
drwxr-xr-x  2 root root  4096 Jun 23 08:15 checkpoints
-rw-r--r--  1 root root  6974 Jun 23 08:15 CLAUDE.md
drwxr-xr-x  2 root root  4096 Jun 23 08:15 configs
drwxr-xr-x  2 root root  4096 Jun 23 08:15 docs
-rw-r--r--  1 root root  4965 Jun 23 08:15 .gitignore
-rw-r--r--  1 root root  1064 Jun 23 08:15 LICENSE
drwxr-xr-x  2 root root  4096 Jun 23 08:15 notebooks
-rw-r--r--  1 root root   203 Jun 23 08:15 pyproject.toml
-rw-r--r--  1 root root 10517 Jun 23 08:15 README.md
-rw-r--r--  1 root root  1225 Jun 23 08:15 requirements.txt
drwxr-xr-x  2 root root  4096 Jun 23 08:15 results
drwxr-xr-x  2 root root  4096 Jun 23 08:15 scripts
drwxr-xr-x  2 root root  4096 Jun 23 08:15

## 2. Зависимости

Kaggle уже содержит PyTorch CUDA. PyG sampler packages (`pyg-lib`, `torch-scatter`, `torch-sparse`) ставятся под текущие версии torch/cuda.

In [7]:
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print("PyG wheel index:", wheel_url)

base_pkgs = [
    "torch_geometric", "xgboost", "scikit-learn", "pandas", "numpy",
    "networkx", "matplotlib", "pyyaml", "tqdm"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *base_pkgs], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyg-lib", "torch-scatter", "torch-sparse", "-f", wheel_url
], check=True)

import torch_geometric
print("torch_geometric:", torch_geometric.__version__)
try:
    import pyg_lib, torch_sparse
    print("sampler deps OK")
except Exception as e:
    print("WARNING: sampler deps import failed:", repr(e))

PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.7 MB/s eta 0:00:00
torch_geometric: 2.8.0
sampler deps OK


## 3. Данные IBM AML

Notebook ищет `HI-Small_Trans.csv` и `HI-Small_Patterns.txt` рекурсивно в `/kaggle/input` и делает symlink/copy в `data/ibm_aml/`.

In [9]:
from pathlib import Path
import os, shutil, glob

def find_one(pattern):
    matches = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Не найден {pattern} в /kaggle/input. Add Data с IBM AML dataset.")
    matches = sorted(matches, key=len)
    print(pattern, "->", matches[0])
    return Path(matches[0])

trans = find_one("HI-Small_Trans.csv")
patterns = find_one("HI-Small_Patterns.txt")

dst_dir = Path("data/ibm_aml")
dst_dir.mkdir(parents=True, exist_ok=True)
for src, name in [(trans, "HI-Small_Trans.csv"), (patterns, "HI-Small_Patterns.txt")]:
    dst = dst_dir / name
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)
    print(dst, "size GB", dst.stat().st_size / 1e9)

!ls -lh data/ibm_aml

HI-Small_Trans.csv -> /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
HI-Small_Patterns.txt -> /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt
data/ibm_aml/HI-Small_Trans.csv size GB 0.475664283
data/ibm_aml/HI-Small_Patterns.txt size GB 0.000323844
total 8.0K
lrwxrwxrwx 1 root root 103 Jun 23 11:36 HI-Small_Patterns.txt -> /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt
lrwxrwxrwx 1 root root 100 Jun 23 11:36 HI-Small_Trans.csv -> /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv


## 4. План проверки

Для полной multi-seed проверки поменяйте `SEEDS = [42, 43, 44]`. На P100 это может занять много часов. По умолчанию запускается только seed 42.

`GPU_SAFE_BATCH=True` снижает batch для PNA до 2048. Это не меняет архитектуру, но может немного изменить обучение из-за другого batching/sampling.

In [22]:
SEEDS = [42]  # для проверки устойчивости: [42, 43, 44]
RUN_GINE = True
RUN_MULTIGNN = True
RUN_PNA = True

# Полный hybrid extraction на 5.08M ребер тяжелый. Включайте отдельным запуском.
RUN_HYBRID_FULL = True

# Smoke-режим для проверки pipeline без полного датасета. Для финальных чисел оставить None.
MAX_ROWS = None  # например 500_000 для smoke

GPU_SAFE_BATCH = True
PNA_BATCH_SIZE = 2048

selected = []
if RUN_GINE: selected.append("ibm_gine_fulldata")
if RUN_MULTIGNN: selected.append("ibm_multignn_fulldata")
if RUN_PNA: selected.append("ibm_pna_fulldata")
print("configs:", selected)
print("seeds:", SEEDS)

configs: ['ibm_gine_fulldata', 'ibm_multignn_fulldata', 'ibm_pna_fulldata']
seeds: [42]


## 5. Генерация временных конфигов

Seed 42 пишет стандартные `output_name`, чтобы `src.compare --ibm` подхватил результаты. Остальные seed пишутся как `<name>_seed<seed>`.

In [11]:
import yaml, copy
from pathlib import Path

TMP_CFG_DIR = Path("configs/kaggle_tmp")
TMP_CFG_DIR.mkdir(parents=True, exist_ok=True)

def make_config(base_name, seed):
    src = Path("configs") / f"{base_name}.yaml"
    with open(src, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["seed"] = int(seed)
    cfg["device"] = "cuda"
    cfg.setdefault("dataset", {})["root"] = "data/ibm_aml"
    if MAX_ROWS is not None:
        cfg["dataset"]["max_rows"] = int(MAX_ROWS)
    cfg["output_name"] = base_name if seed == 42 else f"{base_name}_seed{seed}"
    cfg["checkpoint_dir"] = "checkpoints/kaggle"
    cfg["save_checkpoint"] = True
    if GPU_SAFE_BATCH and base_name == "ibm_pna_fulldata":
        cfg.setdefault("train", {})["batch_size"] = int(PNA_BATCH_SIZE)
    out = TMP_CFG_DIR / f"{base_name}_seed{seed}.yaml"
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    return out

tmp_configs = []
for seed in SEEDS:
    for base in selected:
        tmp_configs.append(make_config(base, seed))

print("generated:")
for p in tmp_configs:
    print(" -", p)

generated:
 - configs/kaggle_tmp/ibm_gine_fulldata_seed42.yaml
 - configs/kaggle_tmp/ibm_multignn_fulldata_seed42.yaml
 - configs/kaggle_tmp/ibm_pna_fulldata_seed42.yaml


## 6. Запуск GNN verification

Если PNA падает с OOM, уменьшите `PNA_BATCH_SIZE` до 1024 и перезапустите генерацию конфигов.

In [12]:
import subprocess, sys, time

for cfg in tmp_configs:
    print("\n" + "="*90)
    print("RUN", cfg)
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "src.train_edge", "--config", str(cfg)], check=True)
    print("elapsed min:", round((time.time() - t0) / 60, 2))


RUN configs/kaggle_tmp/ibm_gine_fulldata_seed42.yaml
[device] cuda
[train] FULL-DATA: все 3,300,881 train-рёбер (pos=2556), pos_weight=100.0
[data] IBM HI-Small: illicit=5177 (0.102%); num_neighbors=[10, 10], batch=2048
[model] gine | params=38,852
epoch  1 | loss 0.2214 | val(sample) AUC-PR 0.1409
epoch  2 | loss 0.1977 | val(sample) AUC-PR 0.1868
epoch  3 | loss 0.1798 | val(sample) AUC-PR 0.1009
epoch  4 | loss 0.1773 | val(sample) AUC-PR 0.2942
epoch  5 | loss 0.1782 | val(sample) AUC-PR 0.2419
epoch  6 | loss 0.1704 | val(sample) AUC-PR 0.2113
epoch  7 | loss 0.1682 | val(sample) AUC-PR 0.1825
epoch  8 | loss 0.1708 | val(sample) AUC-PR 0.1921
epoch  9 | loss 0.1677 | val(sample) AUC-PR 0.2606
early stop на эпохе 9 (best val AUC-PR 0.2942)
[eval] полный val (порог, train-контекст) и полный test (train+val-контекст)...
[VAL ] AUC-PR=0.0333 F1=0.1149
[TEST] AUC-PR=0.0427 F1=0.1199 R@P90=0.0000
[per-pattern recall] fan_out:0.11(16/140), fan_in:0.23(32/137), gather_scatter:0.18(74/40

## 7. Сводка seed-прогонов

In [14]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("results/ibm_*fulldata*_metrics.json")):
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    test = d.get("test_metrics", {}) or {}
    cfg = d.get("config", {}) or {}
    rows.append({
        "file": Path(path).name,
        "output_name": cfg.get("output_name"),
        "seed": cfg.get("seed"),
        "model": d.get("model_type") or (cfg.get("model", {}) or {}).get("type"),
        "auc_pr": test.get("auc_pr"),
        "f1": test.get("f1"),
        "roc_auc": test.get("roc_auc"),
        "r_at_p90": test.get("recall_at_precision_90"),
        "recall": test.get("recall"),
    })

df = pd.DataFrame(rows).sort_values(["auc_pr"], ascending=False, na_position="last")
display(df)
df.to_csv("results/kaggle_gpu_seed_summary.csv", index=False)

,file,output_name,seed,model,auc_pr,f1,roc_auc,r_at_p90,recall
10,ibm_pna_fulldata_metrics.json,ibm_pna_fulldata,42.0,pna,0.059094,0.087187,0.963943,0.0,0.063960
9,ibm_multipna_fulldata_metrics.json,ibm_multipna_fulldata,NaN,pna,0.057400,0.096000,NaN,0.0,NaN
3,ibm_gine_port_fulldata_metrics.json,ibm_gine_port_fulldata,42.0,gine,0.054189,0.111380,0.944262,0.0,0.127920
7,ibm_multignn_fulldata_metrics.json,ibm_multignn_fulldata,42.0,gine,0.051726,0.123328,0.917073,0.0,0.105117
2,ibm_gine_fulldata_metrics.json,ibm_gine_fulldata,42.0,gine,0.042744,0.119900,0.940818,0.0,0.134038
5,ibm_multignn_big_fulldata_metrics.json,ibm_multignn_big_fulldata,42.0,gine,0.042305,0.095675,0.960035,0.0,0.133482
4,ibm_gine_rev_fulldata_metrics.json,ibm_gine_rev_fulldata,42.0,gine,0.040115,0.083142,0.926712,0.0,0.100667
8,ibm_multipna_eu_fulldata_metrics.json,ibm_multipna_eu_fulldata,NaN,pna,0.040100,0.095000,NaN,0.0,NaN
6,ibm_multignn_eu_fulldata_metrics.json,ibm_multignn_eu_fulldata,42.0,gine,0.034646,0.115272,0.947189,0.0,0.185762
0,ibm_gine_ego_fulldata_metrics.json,ibm_gine_ego_fulldata,42.0,gine,0.032672,0.080998,0.931764,0.0,0.265295


## 8. Опционально: полный гибрид GNN embedding -> XGBoost

Это самый тяжелый шаг. Запускайте после проверки, что нужный checkpoint существует. Для Multi-GNN seed 42 путь обычно `checkpoints/kaggle/ibm_multignn_fulldata.pt`.

In [25]:
from pathlib import Path
import subprocess, sys, os

REPO = Path("/kaggle/working/gnn-aml-transaction-graphs")
os.chdir(REPO)

ckpt = Path("checkpoints/kaggle/ibm_multignn_fulldata.pt")
if not ckpt.exists():
    ckpt = Path("checkpoints/ibm_multignn_fulldata.pt")

print("cwd:", Path.cwd())
print("checkpoint:", ckpt, "exists:", ckpt.exists())
assert ckpt.exists(), f"checkpoint not found: {ckpt}"

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO) + ":" + env.get("PYTHONPATH", "")

out_json = "results/ibm_hybrid_gnn_xgb_metrics.json"
subprocess.run(
    [sys.executable, "scripts/hybrid_gnn_xgb.py", "-", str(ckpt), out_json],
    check=True,
    env=env,
)

cwd: /kaggle/working/gnn-aml-transaction-graphs
checkpoint: checkpoints/kaggle/ibm_multignn_fulldata.pt exists: True
[device] cuda
[ckpt] checkpoints/kaggle/ibm_multignn_fulldata.pt rev=True nbr=[10, 10]
[data] edges=5,078,345 pos tr/va/te=2556/823/1798
[emb] извлекаю эмбеддинги/скоры GNN (анти-утечка контексты)...
   batch 0
   batch 200
   batch 400
   batch 600
   batch 800
[emb] train готов
   batch 0
[emb] val готов
   batch 0
   batch 200
[emb] test готов

=== ТЕСТ 1: ГИБРИД GNN->XGBoost ===
  сырьё ребра (raw)                        AUC-PR 0.2893 | F1 0.4121 | ROC 0.961
  полный XGBoost (raw+node+parallel)       AUC-PR 0.2484 | F1 0.2874 | ROC 0.968
  сырьё + GNN-эмбеддинг (ГИБРИД)           AUC-PR 0.3299 | F1 0.3981 | ROC 0.958
  Δ гибрид-сырьё = +0.0406  (>>> ГРАФ ПОМОГ)

=== ТЕСТ 2: КОМПЛЕМЕНТАРНОСТЬ (ловит ли GNN то, что XGBoost пропускает) ===
  budget N=  500: XGB ловит  280/1798 | ансамбль(XGB+GNN)  268 | только-GNN сверх XGB +54
  budget N= 1000: XGB ловит  490/1798 | ан

CompletedProcess(args=['/usr/bin/python3', 'scripts/hybrid_gnn_xgb.py', '-', 'checkpoints/kaggle/ibm_multignn_fulldata.pt', 'results/ibm_hybrid_gnn_xgb_metrics.json'], returncode=0)

## 9. Пересборка итоговых таблиц и графиков

In [26]:
subprocess.run([sys.executable, "-m", "src.compare", "--ibm"], check=True)
print("main artifacts:")
for p in [
    "results/ibm_all_variants.md",
    "results/ibm_all_variants_ranking.png",
    "results/ibm_family_best.png",
    "results/ibm_ablation_heatmap.png",
    "results/ibm_ladder.md",
    "results/per_pattern.md",
]:
    print(p, Path(p).exists())


=== все рассмотренные IBM-варианты ===

[saved] results/ibm_all_variants.csv
[saved] results/ibm_all_variants.md
[saved] results/ibm_all_variants_ranking.png
[saved] results/ibm_family_best.png
[saved] results/ibm_ablation_heatmap.png

| variant | auc_pr | f1 | recall_at_precision_90 | recall |
|---|---|---|---|---|
| XGBoost | 0.1289 | 0.1900 | 0.0000 | 0.2508 |
| GINe (base) | 0.0190 | 0.0562 | 0.0000 | 0.2442 |
| +reverse | 0.0396 | 0.0851 | 0.0000 | 0.1529 |
| +port | 0.0385 | 0.1055 | 0.0000 | 0.1329 |
| +ego | 0.0366 | 0.1077 | 0.0000 | 0.1774 |
| Multi-GNN (full) | 0.0510 | 0.1202 | 0.0000 | 0.1529 |

[saved] results/ibm_comparison.csv
[saved] results/ibm_comparison.md
[saved] results/ablation.png

=== режим без norm_time ===

| variant | auc_pr | f1 | recall_at_precision_90 | recall |
|---|---|---|---|---|
| XGBoost | 0.2398 | 0.2976 | 0.0217 | 0.3103 |
| GINe (base) | 0.0442 | 0.1199 | 0.0000 | 0.1357 |
| +reverse | 0.0255 | 0.0510 | 0.0000 | 0.5028 |
| +port | 0.0354 | 0.100

## 10. Упаковка результатов для скачивания

In [28]:
import zipfile, os
zip_path = Path("/kaggle/working/kaggle_gpu_results_2.zip")
patterns = [
    "results/ibm_*metrics.json",
    "results/ibm_*.md",
    "results/ibm_*.png",
    "results/per_pattern.*",
    "results/kaggle_gpu_seed_summary.csv",
    "checkpoints/kaggle/*.pt",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in patterns:
        for f in glob.glob(pat):
            z.write(f)
print("saved", zip_path, "size MB", zip_path.stat().st_size / 1e6)

saved /kaggle/working/kaggle_gpu_results_2.zip size MB 1.929746
